# Single Object Tracking with Missed Detections. 

Track management faces three primary challenges:

2. Sensors may fail to report a contact, perhaps previously tracked, for some period of time.
1. Sensors may report false positives in the form of noise or objects that are not of interest.
3. Associations between measurements and objects is generally not known.

We will start by considering the single object tracking case, which eliminates the association problem for now. That leaves us with the problems of clutter (false positives) and the chance of missed detections. The latter is what we cover first. 

## Motion Model

$$
x_k = f(x_{k - 1}) + q_{k - 1}, ~~~~~~~ q_{k - 1} \sim \mathcal{N}(0, Q_{k - 1})
$$

Then, assuming a fixed value of random variable $X_k = x_k$, our only source of variability comes from the pocess noise $q_{k - 1}$. This leads to the distribution 

$$
p(x_k | x_{k - 1}) = \mathcal{N}(x_k ; f(x_{k - 1}), Q_{k - 1})
$$

## Measurement Model

Given that an object is detected with probability $P_D(x_k)$, we receive a measurement from a distribution

$$p(o_k | x_k) = g(o_k | x_k)$$

Where we know that $o_k$ is a function of the true state, plus some Gaussian error:

$$
o_k = h(x_k) + v_k,  ~~~~~~~ v_{k} \sim \mathcal{N}(0, R_{k})
$$

And therefore 

$$
g(o_k | x_k) = \mathcal{N}(h(x_k), R_k)
$$

But this is an incomplete model, because it does not account for the possibility of missed detections. 

### Measurement Model with Missed Detections

Define $O_k$ as the observation matrix at time $k$, comprised of observations $o^i_k$ as column vectors. For **single object tracking**, this is defined as:

$$
O_k = 
\begin{cases}
\{\} & \text{if undetected} \\
o_k & \text{if detected}
\end{cases}
$$

We define $|O_k|$ as the number of column vectors in $O_k$. In the single object tracking scenario, the matrix can only have 0 or 1 column vectors:

$$
|O_k| = \begin{cases}
0 & \text{if } O_k = \{\} \\
1 & \text{if } O_k = o_k
\end{cases}
$$ 

Which is naturally described by a Bernoulli distribution, $|O_k| \sim \text{Bernoulli}(P_D(x_t))$, where $P_D$ is the probability of detection for the given true state.

$$
p(|O_k|) = \begin{cases}
1 - P_D(x_k) & \text{if } O_k = \{\} \\
P_D(x_k) & \text{if } O_k = o_k
\end{cases}
$$ 

For example, $P_D(x_k)$ might decrease as an object moves farther from the sensor which is trying to observe it. 

Given this probability model of detection, we can form our likelihood function as

$$
P(O_k | x_k) = 
\begin{cases}
P_D(x_k)g(o_k | x_k) & \text{if } O_k = o_k \\
1 - P_D(x_k)  & \text{if } O_k = \{\}
\end{cases}
$$

## Computing the Posterior

We wish to compute the posterior $p(x_k | o_{1:k})$, and we will split the computation in the standard way by a prediction step and Bayes update step.

### Prediction Step

Prediction is performed in the standard way by integrating the product of the motion model and the posterior from the last time step:

$$
P(x_k | o_{1:k - 1}) = \int p(x_k | x_{k - 1}) p( x_{k - 1} | o_{1 : k - 1}) dx_{k - 1} 
$$

### Update Step

Following bayes Rule and combined with the likelihood we defined above, we have

$$
\begin{align}
\text{posterior} &\propto \text{prior} \times \text{likelihood} \\
&= 
\begin{cases}
p(x_{k} | O_{1:k-1}) P_D(x_k) g(o_k | x_k) & \text{if } O_k = o_k \\
p(x_{k} | O_{1:k-1})(1 - P_D(x_k))  & \text{if } O_k = \{\}
\end{cases}
\end{align}
$$

In many cases P_D(x_k) is assumed constant, and can therefore be removed. But, sometimes P_D(x_k) is not constant and therefore contains information about the state $x_k$. 
